# Final Notebook - House Price Prediction

Notebook này tổng hợp toàn bộ chương trình dự đoán giá nhà: lấy dữ liệu Kaggle, kiểm tra chất lượng dữ liệu, EDA, preprocessing, feature engineering, training, tuning, đánh giá tối ưu, inference demo và giao diện HTML.

## 0. Nội dung được tổng hợp

- Bài toán regression dự đoán `SalePrice`.
- Nguồn dữ liệu Kaggle và cách chạy trên Google Colab.
- Hình ảnh EDA/model giống cấu trúc báo cáo spam email: distribution, scatter, correlation, model comparison, residuals, learning curve.
- Model selection dựa trên RMSE, MAE, R2 và kết quả tuning.

## 1. Problem Definition

Mục tiêu là xây dựng mô hình Machine Learning dự đoán giá bán nhà. Mỗi dòng dữ liệu tương ứng một căn nhà; các cột đầu vào mô tả diện tích, chất lượng, năm xây dựng, garage, khu vực và điều kiện bán; target là `SalePrice`.

Đây là bài toán **supervised learning - regression** vì output là giá trị số liên tục.

## 2. Success Metrics

- **RMSE**: lỗi lớn bị phạt mạnh, phù hợp khi sai số giá nhà lớn là nghiêm trọng.
- **MAE**: dễ giải thích vì là sai số trung bình theo đơn vị tiền.
- **R2**: cho biết model giải thích được bao nhiêu phương sai của `SalePrice`.
- Model cuối được chọn theo RMSE thấp nhất, sau đó kiểm tra MAE/R2 để tránh chọn model mất cân bằng.

## 3. Data Collection - Kaggle và Google Colab

Dataset chính: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

Dataset thay thế:

- https://www.kaggle.com/datasets/yasserh/housing-prices-dataset
- https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset
- https://www.kaggle.com/datasets/quantbruce/real-estate-price-prediction
- https://www.kaggle.com/datasets/amitabhajoy/bengaluru-house-price-data

Trên Google Colab, upload `kaggle.json`, tải competition data, giải nén, rồi đặt `train.csv` vào `data/raw/train.csv`. Nếu không có Kaggle credential, notebook dùng real Ames Housing public data từ OpenML; workflow train chính không dùng sample data.

In [ ]:
# Colab setup tham khảo. Chạy cell này khi mở notebook trên Google Colab.
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle data/raw
# !cp kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle pandas numpy scikit-learn matplotlib seaborn joblib flask
# !kaggle competitions download -c house-prices-advanced-regression-techniques -p data/raw
# !unzip -o data/raw/house-prices-advanced-regression-techniques.zip -d data/raw

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## 4. Data Loading

Loader ưu tiên `data/processed/house_prices_clean_labeled.csv`, được tạo từ `data/raw/train.csv` của Kaggle hoặc real OpenML Ames Housing data. Sample data không được dùng trong workflow train chính.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from config import FIGURES_DIR, METRICS_PATH, MODEL_PATH, TARGET_COLUMN
from src.data_loader import load_dataset, resolve_dataset_path, split_features_target
from src.data_quality import summarize_quality
from src.data_pipeline import prepare_real_dataset
from src.feature_engineering import add_house_features
from src.predict import predict_price
from src.train import train_all

prepared_df = prepare_real_dataset()
dataset_path = resolve_dataset_path()
df = load_dataset(dataset_path)
print('Dataset path:', dataset_path)
print('Shape:', df.shape)
df.head()

## 5. EDA - Phân tích dữ liệu

EDA tập trung vào `SalePrice`, quan hệ giữa diện tích và giá, cùng các feature numeric có tương quan mạnh với giá nhà.

In [ ]:
df[TARGET_COLUMN].describe()

### Hình 1 - SalePrice distribution

![SalePrice distribution](../reports/figures/saleprice_distribution.png)

### Hình 2 - Scatter feature vs SalePrice

![GrLivArea vs SalePrice](../reports/figures/grlivarea_vs_saleprice.png)

### Hình 3 - Correlation heatmap

![Correlation heatmap](../reports/figures/saleprice_correlation_heatmap.png)

## 6. Data Quality

Kiểm tra missing values, duplicate rows, numeric/categorical columns và target missing. Đây là bước bắt buộc trước khi preprocessing.

In [ ]:
quality_report = summarize_quality(df)
quality_report

## 7. Preprocessing

Pipeline dùng `ColumnTransformer` để xử lý dữ liệu đúng cách:

- Numeric: median imputation + standard scaling.
- Categorical: most frequent imputation + one-hot encoding.
- `handle_unknown='ignore'` giúp inference không lỗi khi gặp category mới.

## 8. Feature Engineering

Các feature bổ sung:

- `HouseAgeAtRemodel = YearRemodAdd - YearBuilt`.
- `AreaPerBedroom = GrLivArea / BedroomAbvGr`.
- `GarageAreaPerCar = GarageArea / GarageCars`.

Các feature này chỉ được tạo nếu cột nguồn tồn tại, nên notebook vẫn chạy được với dataset thay thế.

In [ ]:
df_features = add_house_features(df)
X, y = split_features_target(df_features)
print('Feature shape:', X.shape)
print('Target shape:', y.shape)
X.head()

## 9. Model Training

Các model được train:

- Linear Regression baseline.
- Ridge Regression.
- Lasso Regression.
- Random Forest Regressor.
- Gradient Boosting Regressor.

Sau đó notebook chạy thêm tuning bằng cross-validation cho Ridge, Lasso, Random Forest và Gradient Boosting.

In [ ]:
result = train_all(dataset_path=dataset_path, save_artifacts=True)
metrics = result['metrics']
tuning_report = result['tuning_report']
print('Best model:', result['best_model_name'])
print('Optimization status:', result['optimization_status'])
metrics

## 10. Model Metrics - So sánh kết quả

Bảng metric được lưu tại `reports/metrics/model_metrics.csv`. Hình dưới giúp so sánh RMSE giữa các model.

![Model RMSE comparison](../reports/figures/model_rmse_comparison.png)

In [ ]:
pd.read_csv(METRICS_PATH)

### Nhận xét so sánh model

- Model có RMSE thấp nhất được chọn làm model cuối.
- Kết quả hiện tại được train trên real filtered data, không dùng sample dataset.
- Nếu tuned model không vượt baseline trên test set, kết luận là baseline hiện đang phù hợp hơn với dữ liệu đang dùng.

## 11. Hyperparameter Tuning và kiểm tra tối ưu

Tuning dùng cross-validation với scoring `neg_root_mean_squared_error`. Báo cáo này trả lời yêu cầu kiểm tra model đã tối ưu chưa.

In [ ]:
tuning_report

## 12. Actual vs Predicted

Hình này tương tự phần đánh giá model của spam email: thay vì confusion matrix, bài toán regression dùng biểu đồ actual-vs-predicted.

![Actual vs predicted](../reports/figures/actual_vs_predicted_saleprice.png)

## 13. Residual Analysis

Residual giúp kiểm tra model còn sai lệch hệ thống hay không. Nếu residual phân bố quanh 0 và không tạo pattern rõ, model ổn hơn.

![Residual distribution](../reports/figures/residual_distribution.png)

![Residuals vs predicted](../reports/figures/residuals_vs_predicted.png)

## 14. Learning Curve

Learning curve cho biết khi tăng số lượng dữ liệu train, train RMSE và validation RMSE thay đổi thế nào.

![Learning curve](../reports/figures/learning_curve_best_model.png)

### Nhận xét learning curve

- Train RMSE thấp nhưng validation RMSE cao: model có dấu hiệu overfitting.
- Cả train và validation RMSE cao: model underfitting hoặc thiếu feature tốt.
- Hai đường hội tụ ở mức thấp: model ổn hơn.
- Kết luận cuối nên dựa trên real Kaggle/OpenML data đã lọc; nếu thay dataset, cần chạy lại toàn bộ pipeline để sinh lại metric và figures.

## 15. Model Selection

Model cuối được lưu vào `models/house_price_pipeline.joblib`. Artifact này chứa pipeline preprocessing, model tốt nhất, danh sách feature và metric để inference dùng đúng cấu trúc training.

In [ ]:
print('Saved model:', MODEL_PATH)
print('Best model:', result['best_model_name'])
print('Optimization status:', result['optimization_status'])

## 16. Demo Predict nhà mới

Demo lấy một dòng input, bỏ target, rồi predict `SalePrice` bằng pipeline đã lưu.

In [ ]:
demo_input = X.head(1)
prediction = predict_price(demo_input)
pd.DataFrame({'PredictedSalePrice': prediction})

## 17. HTML UI dự đoán giá nhà

Giao diện HTML nằm ở `templates/index.html`, backend Flask nằm ở `app_html.py`.

Chạy giao diện:

```powershell
python app_html.py
```

Sau đó mở `http://127.0.0.1:5000` để nhập thông tin nhà và nhận giá dự đoán.

## 18. Code Standard và kiểm soát chất lượng

- Code được tách module theo `src/`: loader, quality, preprocessing, feature engineering, train, evaluate, predict.
- Test nằm trong `tests/`.
- Notebook parse bằng JSON chuẩn.
- Model, metrics và figures đều được sinh tự động khi chạy `python -m src.train`.

## 19. Kết luận cuối

Dự án đã có đủ workflow Machine Learning từ data collection đến deployment giao diện HTML. Với real Ames/House Prices data đã lọc, model tốt nhất hiện tại là model có RMSE thấp nhất trong bảng metrics; notebook sẽ tự cập nhật kết luận khi chạy lại bằng Kaggle `train.csv`.

## 20. Checklist trình bày

- Có link Kaggle và hướng dẫn Colab.
- Có data collection và data quality report.
- Có lọc dữ liệu rác, lọc label và đánh nhãn dữ liệu.
- Có EDA figures.
- Có preprocessing và feature engineering.
- Có model comparison và hyperparameter tuning.
- Có residual analysis và learning curve.
- Có demo prediction.
- Có HTML UI dùng model đã train.